# Mesurer et accélérer des traitements Spark sur les transactions immobilières

Ce TP reprend les transactions immobilières du TP d'analyse avec Spark, cette fois pour mesurer ce que coûte chaque traitement et le rendre plus rapide. Vous suivez une requête dans l'interface de Spark, lisez son plan physique, réglez les partitions du shuffle, repérez une clé déséquilibrée et salez-la, choisissez la stratégie d'une jointure, mettez en cache ce qui se relit, comparez une UDF Python aux fonctions de Spark et écrivez les données pour la lecture suivante. Pour finir, vous diagnostiquez une requête lente, symptôme par symptôme, et la réécrivez.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les deux cœurs de la machine de Colab, et les données tiennent sur une machine : les durées sont des ordres de grandeur, qui varient d'un essai à l'autre. Les écarts, eux, se retrouvent sur un cluster, et grandissent avec les données.

## Installation et données

Les données sont les [demandes de valeurs foncières géolocalisées](https://www.data.gouv.fr/fr/datasets/demandes-de-valeurs-foncieres-geolocalisees/) (Etalab, Licence Ouverte 2.0), dans la version retravaillée du dépôt [NyxAether/DVF](https://github.com/NyxAether/DVF) : les transactions immobilières en France de 2014 à 2022, une ligne par transaction. Elles forment un dossier Parquet de 635 Mo, `dvf/dvf-linearized-2014-2022.parquet`, partitionné par année : un sous-dossier par valeur de `annee_mutation`, qui tient un fichier.

La table `departements-region.csv` ([data.gouv.fr](https://www.data.gouv.fr/datasets/departements-et-leurs-regions), domaine public) donne la région de chaque département : `num_dep`, `dep_name` et `region_name`.

In [ ]:
!pip install pyspark
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh dvf
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/france-departements/departements-region.csv

In [ ]:
import json
import os
import pathlib
import sys
import time
import urllib.request

import pandas as pd
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf

La session tourne en local sur deux cœurs, avec 4 Go de mémoire pour le pilote (1 Go par défaut). Les barres de progression de Spark sont masquées : elles encombreraient les sorties. Les processus Python que Spark lance pour exécuter du code Python (exercice 8) prennent l'interpréteur de ce notebook, `sys.executable`, qui a pandas et pyarrow : hors de Colab, Spark prendrait sinon le `python3` du `PATH`, qui peut ne pas les avoir.

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
spark = (
  SparkSession.builder.appName("dvf-performances")
  .master("local[2]")
  .config("spark.driver.memory", "4g")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
sc = spark.sparkContext
print(spark.version)

## Outils de mesure

L'interface web de Spark (port 4040) n'est pas joignable depuis votre navigateur sur Colab, mais son API REST répond au noyau du notebook. `api` en lit une page en JSON. `etapes` affiche les dernières étapes de l'application, de la plus récente à la plus ancienne : leur état (`SKIPPED` pour une étape dont la sortie existait déjà), leur nombre de tâches, leur temps de calcul cumulé en millisecondes et les octets écrits par leur shuffle. `taches` donne les quantiles (minimum, médiane, maximum) de la durée des tâches d'une étape et du nombre de lignes que chacune a lues du shuffle.

`mesurer` appelle trois fois une fonction et affiche la meilleure durée. Spark garde les fichiers de shuffle d'un `DataFrame` : relancer une action sur le même `DataFrame` saute les étapes déjà faites et ne mesure plus rien. La fonction à mesurer construit donc sa requête à chaque appel, comme `lambda: dvf.groupBy(...).count().collect()`.

In [ ]:
def api(chemin):
  url = f"{sc.uiWebUrl}/api/v1/applications/{sc.applicationId}/{chemin}"
  return json.load(urllib.request.urlopen(url))


def etapes(n=3):
  colonnes = ["stageId", "status", "numTasks", "executorRunTime", "shuffleWriteBytes"]
  return pd.DataFrame(api("stages"))[colonnes].head(n)


def taches(etape):
  resume = api(f"stages/{etape}/0/taskSummary?quantiles=0,0.5,1")
  return {
    "duree_ms": resume["executorRunTime"],
    "lignes_lues": resume["shuffleReadMetrics"]["readRecords"],
  }


def mesurer(fonction, essais=3):
  durees = []
  for _ in range(essais):
    debut = time.time()
    fonction()
    durees.append(time.time() - debut)
  print(f"{min(durees):.2f} s")
  return min(durees)

## 1. Suivre une requête

Lisez le dossier Parquet dans `dvf`. En combien de partitions Spark le lit-il ? Calculez ensuite la valeur foncière moyenne (`valeur_fonciere`) par département (`code_departement`), ramenée au pilote par `collect`, puis affichez les étapes qu'elle a lancées avec `etapes()`.

Combien de tâches chaque étape a-t-elle eues ? Laquelle a écrit un shuffle, et pourquoi ? Relancez la même action sur le même `DataFrame` et réaffichez les étapes : que constatez-vous ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

`getNumPartitions` est une méthode du RDD sous-jacent, `dvf.rdd`. Les fichiers font de 33 à 94 Mo : comparez au plafond de 128 Mo par partition (`spark.sql.files.maxPartitionBytes`). Gardez le `DataFrame` de la moyenne dans une variable pour relancer la même action.

```python
dvf = spark.read.parquet(...)
print(dvf.rdd.getNumPartitions())  # mode classique ; pas de .rdd sous Spark Connect
moyennes = dvf.groupBy(...).agg(F.avg(...))
moyennes.collect()
print(etapes())
moyennes.collect()
print(etapes())
```

### Solution

In [ ]:
dvf = spark.read.parquet("dvf/dvf-linearized-2014-2022.parquet")
print(dvf.rdd.getNumPartitions())  # mode classique ; pas de .rdd sous Spark Connect
moyennes = dvf.groupBy("code_departement").agg(F.avg("valeur_fonciere"))
moyennes.collect()
print(etapes())
moyennes.collect()
print(etapes())
# 8 partitions pour 9 fichiers de 33 à 94 Mo : Spark regroupe les petits fichiers
# dans des partitions de 128 Mo au plus. L'étape 1 a 8 tâches, une par partition :
# elles lisent les données, calculent des moyennes partielles par département et
# écrivent le shuffle (54 Ko), car la moyenne d'un département demande ses lignes de
# toutes les partitions. L'étape 3, une seule tâche (l'exécution adaptative a
# regroupé les partitions du shuffle), calcule les moyennes finales. L'étape 2 est
# l'étape 1 vue par le job suivant, déjà faite : SKIPPED. Relancée sur le même
# DataFrame, l'action ne refait que la dernière étape : Spark a gardé les fichiers
# du shuffle. C'est pourquoi mesurer reconstruit la requête à chaque essai.

## 2. Le plan physique

Voici cinq filtres sur `dvf`, que vous classez d'après le plan de `dvf.filter(filtre).select("valeur_fonciere")`, affiché avec `explain("formatted")` :

- `F.col("valeur_fonciere") > 1e6` ;
- `F.col("code_postal").startswith("75")` ;
- `F.substring("code_postal", 1, 2) == "75"` ;
- `F.col("annee_mutation") >= 2020` ;
- `F.col("mois_mutation").isin(7, 8)`.

Pour chacun, dans le bloc `Scan parquet` : est-il dans `PartitionFilters` (des dossiers écartés), dans `PushedFilters` (confié au lecteur Parquet), ou dans aucun des deux (appliqué après la lecture) ? Que lit `ReadSchema` ? Comptez aussi les lignes que chaque filtre garde.

Mesurez enfin avec `mesurer` la moyenne de `valeur_fonciere` sous le filtre le plus sélectif (celui qui garde le moins de lignes), puis sous le moins sélectif. Lequel est le plus rapide, et pourquoi ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Rangez les filtres dans un dictionnaire, nom → colonne booléenne, et parcourez-le. Le type d'une colonne est dans `dvf.printSchema()`, ou dans le plan : `code_postal` n'est pas une chaîne. Un champ vide n'est pas affiché dans le plan formaté. `mesurer` prend une fonction sans argument, qui construit la requête et lance l'action : `lambda: dvf.filter(...).agg(F.avg(...)).collect()`.

```python
filtres = {
  "valeur > 1e6": F.col("valeur_fonciere") > 1e6,
  "code postal startswith": ...,
  "code postal substring": ...,
  "annee >= 2020": ...,
  "mois isin": ...,
}
for nom, filtre in filtres.items():
  print(nom, dvf.filter(filtre).count())
  dvf.filter(filtre).select("valeur_fonciere").explain("formatted")
for nom in [..., ...]:
  print(nom, end=" ")
  mesurer(lambda: dvf.filter(filtres[nom]).agg(...).collect())
```

### Solution

In [ ]:
filtres = {
  "valeur > 1e6": F.col("valeur_fonciere") > 1e6,
  "code postal startswith": F.col("code_postal").startswith("75"),
  "code postal substring": F.substring("code_postal", 1, 2) == "75",
  "annee >= 2020": F.col("annee_mutation") >= 2020,
  "mois isin": F.col("mois_mutation").isin(7, 8),
}
for nom, filtre in filtres.items():
  print(nom, dvf.filter(filtre).count())
  dvf.filter(filtre).select("valeur_fonciere").explain("formatted")
for nom in ["valeur > 1e6", "annee >= 2020"]:
  print(nom, end=" ")
  mesurer(lambda: dvf.filter(filtres[nom]).agg(F.avg("valeur_fonciere")).collect())
# valeur > 1e6 : PushedFilters (GreaterThan), 117 082 lignes. mois isin :
# PushedFilters (In), 1 773 556 lignes. annee >= 2020 : PartitionFilters, seuls les
# dossiers de 2020 à 2022 sont ouverts, 3 314 678 lignes. Les deux filtres sur
# code_postal ne sont pas poussés (PushedFilters ne garde que IsNotNull) :
# code_postal est un entier, que Spark convertit en chaîne avant de comparer, et
# un filtre sur une colonne calculée s'applique après la lecture. Ils gardent
# 332 385 lignes, dont celles du 07500 (Ardèche), devenu 7500 : sur un entier,
# between(75000, 75999) est poussé, et juste. ReadSchema : une ou deux colonnes,
# celle du select et celle du filtre (annee_mutation vient du nom des dossiers).
# Le plus sélectif est le plus lent, environ trois fois plus : ses valeurs
# foncières sont réparties dans tous les groupes de lignes, dont le minimum et le
# maximum n'en écartent presque aucun, alors que le filtre de partition ne lit que
# trois fichiers sur neuf. Ce qui compte, c'est ce que le lecteur peut sauter.

## 3. Les partitions du shuffle

Désactivez l'exécution adaptative (`spark.sql.adaptive.enabled` à `"false"`), mesurez la moyenne de `valeur_fonciere` par département, et affichez les deux dernières étapes : combien de tâches la seconde étape a-t-elle, pour combien de départements ? Passez `spark.sql.shuffle.partitions` à `"8"` et mesurez de nouveau.

Remettez `spark.sql.shuffle.partitions` à sa valeur par défaut (`spark.conf.unset`), réactivez l'exécution adaptative, mesurez une troisième fois, et affichez les trois dernières étapes. Quel opérateur du plan final (`explain()` après l'action) a changé le nombre de partitions ?

**Documentation utile**

- [pyspark.sql.conf.RuntimeConfig](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.conf.RuntimeConfig.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`spark.conf.set(cle, valeur)` change une option SQL en cours de session, `spark.conf.unset(cle)` lui rend sa valeur par défaut. Le plan adaptatif n'est final qu'après une action : appelez `collect()` sur le `DataFrame`, puis son `explain()`.

```python
spark.conf.set("spark.sql.adaptive.enabled", "false")


def moyenne_par_departement():
  return dvf.groupBy(...).agg(...).collect()


mesurer(moyenne_par_departement)
print(etapes(2))
spark.conf.set("spark.sql.shuffle.partitions", ...)
mesurer(moyenne_par_departement)
spark.conf.unset(...)
spark.conf.set("spark.sql.adaptive.enabled", ...)
mesurer(moyenne_par_departement)
print(etapes())
moyennes = dvf.groupBy(...).agg(...)
moyennes.collect()
moyennes.explain()
```

### Solution

In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", "false")


def moyenne_par_departement():
  return dvf.groupBy("code_departement").agg(F.avg("valeur_fonciere")).collect()


mesurer(moyenne_par_departement)
print(etapes(2))
spark.conf.set("spark.sql.shuffle.partitions", "8")
mesurer(moyenne_par_departement)
spark.conf.unset("spark.sql.shuffle.partitions")
spark.conf.set("spark.sql.adaptive.enabled", "true")
mesurer(moyenne_par_departement)
print(etapes())
moyennes = dvf.groupBy("code_departement").agg(F.avg("valeur_fonciere"))
moyennes.collect()
moyennes.explain()
# Sans exécution adaptative : 200 tâches (spark.sql.shuffle.partitions) pour 97
# départements, la plupart sans aucune ligne, en 1,7 s environ ; avec 8 partitions,
# 1,1 s. Avec l'exécution adaptative : 0,9 s et une seule tâche après le shuffle.
# AQEShuffleRead coalesced a regroupé les 200 partitions que prévoyait le plan
# initial (Exchange hashpartitioning(code_departement, 200)).

## 4. Une clé déséquilibrée

Comptez les transactions par nature (`nature_mutation`). Répartissez ensuite `dvf` en 8 partitions selon `nature_mutation`, dans `par_nature`, et comptez les lignes de chaque partition avec `F.spark_partition_id()`.

Comptez les parcelles distinctes (`id_parcelle`) par nature et par département sur `par_nature`, en mesurant la durée (un seul essai suffit). Affichez les étapes, puis, avec `taches`, les quantiles de l'étape qui a lu le shuffle (la plus récente des étapes de 8 tâches terminées) : que montre l'écart entre la médiane et le maximum ? L'exécution adaptative est active depuis l'exercice 3 : que dit l'`Exchange` de `par_nature` dans le plan final (`explain()` après l'action), et pourquoi n'a-t-elle ni regroupé ni rééquilibré ces partitions ?

Salez enfin la clé : ajoutez à `dvf` une colonne `sel`, `F.abs(F.hash("id_parcelle")) % 8`, et répartissez en 8 partitions selon la nature et le sel. Comptez les parcelles distinctes par nature, département et sel, puis sommez ces comptes sur le sel : retrouvez-vous les comptes de `par_nature` ? Comparez les quantiles des tâches. Pourquoi le sel se calcule-t-il ici à partir de `id_parcelle`, et pas au hasard ?

**Documentation utile**

- [pyspark.sql.functions.count_distinct](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.count_distinct.html)
- [pyspark.sql.functions.hash](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.hash.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.spark_partition_id()` donne le numéro de la partition de chaque ligne : regroupez sur lui. `F.count_distinct` compte les valeurs distinctes. `etapes()` rend un `DataFrame` pandas : filtrez ses lignes sur `status` et `numTasks`, la première qui reste est la plus récente ; son `stageId` est le numéro à passer à `taches`. `repartition` prend un nombre de partitions puis une ou plusieurs colonnes. Un `DataFrame` relancé sauterait ses étapes : mesurez l'action qui ramène son résultat, gardé dans une liste, puis affichez son plan final. Deux résultats sont les mêmes quand leurs lignes forment le même ensemble (`set`).

```python
dvf.groupBy(...).count().orderBy(F.desc("count")).show(truncate=False)
par_nature = dvf.repartition(8, ...)
par_nature.groupBy(F.spark_partition_id().alias("partition")).count().orderBy(
  "partition"
).show()


def etape_lue():
  liste = etapes(5)
  terminees = liste[(liste["status"] == "COMPLETE") & (liste["numTasks"] == 8)]
  return int(terminees["stageId"].iloc[0])


comptes = par_nature.groupBy(..., ...).agg(F.count_distinct(...).alias("parcelles"))
resultats = []
mesurer(lambda: resultats.append(comptes.collect()), essais=1)
print(etapes())
print(taches(etape_lue()))
comptes.explain()
par_sel = dvf.withColumn("sel", ...).repartition(8, ..., ...)
salees = (
  par_sel.groupBy(..., ..., ...)
  .agg(F.count_distinct(...).alias("parcelles"))
  .groupBy(..., ...)
  .agg(F.sum(...).alias("parcelles"))
)
mesurer(lambda: resultats.append(salees.collect()), essais=1)
print(taches(etape_lue()))
print(set(resultats[0]) == set(resultats[1]))
```

### Solution

In [ ]:
dvf.groupBy("nature_mutation").count().orderBy(F.desc("count")).show(truncate=False)
par_nature = dvf.repartition(8, "nature_mutation")
par_nature.groupBy(F.spark_partition_id().alias("partition")).count().orderBy(
  "partition"
).show()


def etape_lue():
  liste = etapes(5)
  terminees = liste[(liste["status"] == "COMPLETE") & (liste["numTasks"] == 8)]
  return int(terminees["stageId"].iloc[0])


comptes = par_nature.groupBy("nature_mutation", "code_departement").agg(
  F.count_distinct("id_parcelle").alias("parcelles")
)
resultats = []
mesurer(lambda: resultats.append(comptes.collect()), essais=1)
print(etapes())
print(taches(etape_lue()))
comptes.explain()
par_sel = dvf.withColumn("sel", F.abs(F.hash("id_parcelle")) % 8).repartition(
  8, "nature_mutation", "sel"
)
salees = (
  par_sel.groupBy("nature_mutation", "code_departement", "sel")
  .agg(F.count_distinct("id_parcelle").alias("parcelles"))
  .groupBy("nature_mutation", "code_departement")
  .agg(F.sum("parcelles").alias("parcelles"))
)
mesurer(lambda: resultats.append(salees.collect()), essais=1)
print(taches(etape_lue()))
print(set(resultats[0]) == set(resultats[1]))
# 91 % des transactions sont des ventes. Six natures sur 8 partitions : une partition de
# 9,9 millions de lignes, deux vides. Les tâches de l'étape qui compte les parcelles
# durent de quelques millisecondes à une dizaine de secondes, pour une médiane de
# quelques centaines de millisecondes : l'étape attend la tâche des ventes, pendant que
# l'autre cœur ne fait rien. Le déséquilibre vient du repartition par une clé
# déséquilibrée, pas du compte distinct : lancé directement sur dvf, il redistribue par
# nature, département et parcelle, en tâches égales. Le plan marque l'Exchange
# REPARTITION_BY_NUM : un nombre de partitions imposé, qu'AQE ne regroupe pas, et AQE ne
# coupe une partition trop grosse que dans une jointure (skewJoin) ou un rebalance,
# jamais dans une agrégation. Salées, les partitions tiennent entre 20 000 et 2,5
# millions de lignes (des sels tombent dans la même partition), la tâche la plus longue
# dure deux à trois fois moins, et la médiane monte à quelques secondes : les ventes
# sont réparties sur plusieurs tâches. Sur deux cœurs, la durée totale gagne peu (le sel
# ajoute un hachage et une seconde agrégation) ; sur un cluster de dizaines de cœurs,
# c'est la tâche la plus longue qui fixe la durée. Les comptes sont les mêmes (True) :
# une parcelle a toujours le même sel, donc ses lignes restent ensemble et la somme des
# comptes distincts ne la compte qu'une fois. Un sel au hasard répartirait les lignes
# d'une même parcelle entre plusieurs sels : il convient à un count ou à une somme, pas
# à un compte distinct.

## 5. Joindre les régions

Lisez `departements-region.csv` (avec son en-tête) dans `regions`. Avant de joindre, vérifiez les clés : combien de transactions de `dvf` n'ont pas de correspondance quand on compare `code_departement` à `num_dep` ? Affichez quelques codes concernés et comparez-les à ceux de la table.

Ajoutez à `regions` une colonne `code_departement` qui écrit chaque code comme `dvf`, et vérifiez qu'aucune transaction ne reste sans région. Joignez alors `dvf` et `regions` sur `code_departement` dans `avec_region`.

Mesurez la moyenne de `valeur_fonciere` par région (`region_name`), et affichez son plan final (`explain()` après l'action) : quelle jointure Spark a-t-il choisie ? Désactivez ensuite la diffusion au moment du plan (`spark.sql.autoBroadcastJoinThreshold` à `"-1"`), mais laissez-la à l'exécution adaptative, qui a son propre seuil (`spark.sql.adaptive.autoBroadcastJoinThreshold`, à `"10m"`). Mesurez de nouveau, et comparez le plan initial au plan final : quelle jointure chacun montre-t-il, et pourquoi la durée ne revient-elle pas à celle de la première mesure ? Remettez les deux seuils par défaut.

**Documentation utile**

- [pyspark.sql.DataFrame.join](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.join.html)
- [Convertir une tri-fusion en diffusion (AQE)](https://spark.apache.org/docs/latest/sql-performance-tuning.html#converting-sort-merge-join-to-broadcast-join)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une jointure `"left_anti"` garde les lignes de gauche qui n'ont pas de correspondance. Les codes de la table ont deux chiffres (`"01"`) là où DVF en a un (`"1"`) ; `F.regexp_replace(colonne, "^0", "")` retire un zéro initial. Ne prenez pas `F.lpad` du côté de DVF : il tronque aussi `"971"` en `"97"`. Le plan initial s'affiche avant l'action, ou sous `Initial Plan` après elle.

```python
regions = spark.read.csv(..., header=True)
sans_region = dvf.join(regions, dvf.code_departement == regions.num_dep, ...)
print(sans_region.count())
sans_region.select("code_departement").distinct().orderBy("code_departement").show()
regions = regions.withColumn("code_departement", F.regexp_replace(...))
print(dvf.join(regions, "code_departement", ...).count())
avec_region = dvf.join(regions, "code_departement")


def moyenne_par_region():
  return dvf.join(regions, "code_departement").groupBy(...).agg(...).collect()


def par_region():
  return avec_region.groupBy(...).agg(...)


mesurer(moyenne_par_region)
moyennes = par_region()
moyennes.collect()
moyennes.explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", ...)
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", ...)
...
spark.conf.unset(...)
spark.conf.unset(...)
```

### Solution

In [ ]:
regions = spark.read.csv("departements-region.csv", header=True)
sans_region = dvf.join(regions, dvf.code_departement == regions.num_dep, "left_anti")
print(sans_region.count())
sans_region.select("code_departement").distinct().orderBy("code_departement").show()
regions = regions.withColumn("code_departement", F.regexp_replace("num_dep", "^0", ""))
print(dvf.join(regions, "code_departement", "left_anti").count())
avec_region = dvf.join(regions, "code_departement")


def moyenne_par_region():
  return (
    dvf.join(regions, "code_departement")
    .groupBy("region_name")
    .agg(F.avg("valeur_fonciere"))
    .collect()
  )


def par_region():
  return avec_region.groupBy("region_name").agg(F.avg("valeur_fonciere"))


mesurer(moyenne_par_region)
moyennes = par_region()
moyennes.collect()
moyennes.explain()
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", "10m")
mesurer(moyenne_par_region)
moyennes = par_region()
moyennes.collect()
moyennes.explain()
spark.conf.unset("spark.sql.autoBroadcastJoinThreshold")
spark.conf.unset("spark.sql.adaptive.autoBroadcastJoinThreshold")
# 717 110 transactions sans région : les départements 1 à 9, que DVF écrit sans zéro
# initial (« 1 ») et la table avec (« 01 »). Une jointure interne les aurait perdues
# sans erreur. Après regexp_replace, aucune. Par défaut, la table fait quelques Ko :
# BroadcastHashJoin dès le plan initial. Seuil du plan à -1 : le plan
# initial prévoit un SortMergeJoin, qui redistribue et trie les deux tables (deux
# Exchange, deux Sort). Après les deux shuffles, AQE voit que la table des régions
# est petite et la change en BroadcastHashJoin dans le plan final, lu en local
# (AQEShuffleRead local) : les tris sont évités, pas le shuffle de DVF, déjà écrit.
# D'où une durée double environ, presque celle d'une tri-fusion : une diffusion
# décidée au plan
# reste la meilleure, et celle d'AQE rattrape une table qui s'avère petite à
# l'exécution (après un filtre que l'optimiseur ne savait pas estimer).

## 6. Mettre en cache

Gardez dans `ventes` les ventes (`nature_mutation` vaut `"Vente"`) de `avec_region` dont la surface bâtie (`surface_reelle_bati_total`) est positive, avec cinq colonnes : `region_name`, `annee_mutation`, `nombre_pieces_principales`, `surface_reelle_bati_total` et le prix au mètre carré, `prix_m2`.

Écrivez une fonction `trois_moyennes` qui calcule la moyenne de `prix_m2` par région, par année et par nombre de pièces. Mesurez-la (deux essais), puis mettez `ventes` en cache, remplissez le cache avec `count` (en mesurant cette action, un essai) et mesurez de nouveau `trois_moyennes`. Combien de mémoire le cache occupe-t-il (`api("storage/rdd")`, champ `memoryUsed`) ? Libérez-le à la fin.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le prix au mètre carré est `F.col("valeur_fonciere") / F.col("surface_reelle_bati_total")`, nommé par `alias`. `cache()` renvoie le `DataFrame` lui-même et ne calcule rien : c'est la première action qui remplit le cache. `api("storage/rdd")` renvoie une liste, un élément par `DataFrame` en cache.

```python
ventes = avec_region.filter(...).select(
  "region_name",
  ...,
  (F.col(...) / F.col(...)).alias("prix_m2"),
)


def trois_moyennes():
  for colonne in [...]:
    ventes.groupBy(colonne).agg(F.avg("prix_m2")).collect()


mesurer(trois_moyennes, essais=2)
ventes.cache()
mesurer(ventes.count, essais=1)
mesurer(trois_moyennes, essais=2)
memoire = [cache["memoryUsed"] / 2**20 for cache in api(...)]
ventes.unpersist()
print(memoire)
```

### Solution

In [ ]:
ventes = avec_region.filter(
  "nature_mutation = 'Vente' AND surface_reelle_bati_total > 0"
).select(
  "region_name",
  "annee_mutation",
  "nombre_pieces_principales",
  "surface_reelle_bati_total",
  (F.col("valeur_fonciere") / F.col("surface_reelle_bati_total")).alias("prix_m2"),
)


def trois_moyennes():
  for colonne in ["region_name", "annee_mutation", "nombre_pieces_principales"]:
    ventes.groupBy(colonne).agg(F.avg("prix_m2")).collect()


mesurer(trois_moyennes, essais=2)
ventes.cache()
mesurer(ventes.count, essais=1)
mesurer(trois_moyennes, essais=2)
memoire = [cache["memoryUsed"] / 2**20 for cache in api("storage/rdd")]
ventes.unpersist()
print(memoire)
# Sans cache, chaque moyenne relit Parquet et refait la jointure : 5 s environ pour les
# trois. Remplir le cache coûte 6 s (tout le calcul, une fois) ; les trois moyennes
# passent ensuite à 2 s environ. Le cache occupe 97 Mo pour 7,1 millions de ventes et
# cinq colonnes ; avec toutes les colonnes, plus de 1 Go. Pour trois lectures, le cache
# ne paie pas encore : il paie pour un DataFrame relu souvent, ou cher à calculer.

## 7. Cache et apprentissage

Gardez les ventes de 2022 de `ventes`, assemblez leur surface et leur nombre de pièces dans une colonne `features` (`VectorAssembler`), et gardez `features` et `prix_m2` dans `X`.

Choisissez par validation croisée (`CrossValidator`, 3 plis) le `regParam` d'une régression linéaire qui prédit `prix_m2` (20 itérations au plus, solveur `"l-bfgs"`), entre `0.1` et `1.0` (`ParamGridBuilder`), d'après l'erreur quadratique moyenne (`RegressionEvaluator`). Mesurez la durée (un essai), et affichez l'erreur moyenne de chaque réglage (`avgMetrics`).

Mettez ensuite `X` en cache, remplissez-le, et mesurez de nouveau la validation croisée. Combien de modèles a-t-elle entraînés, et d'où vient l'écart ?

**Documentation utile**

- [pyspark.ml.tuning.CrossValidator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.CrossValidator.html)
- [pyspark.ml.tuning.ParamGridBuilder](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.tuning.ParamGridBuilder.html)
- [pyspark.ml.evaluation.RegressionEvaluator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les classes s'importent de `pyspark.ml.tuning` et `pyspark.ml.evaluation`. `ParamGridBuilder().addGrid(regression.regParam, [...]).build()` donne la liste des réglages ; `RegressionEvaluator(labelCol=...)` mesure par défaut la racine de l'erreur quadratique moyenne (`rmse`). `mesurer` rend la durée, pas le modèle : une fonction qui entraîne et range le modèle dans une liste le garde.

```python
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

X = (
  VectorAssembler(inputCols=[...], outputCol="features")
  .transform(ventes.filter(...))
  .select("features", "prix_m2")
)
regression = LinearRegression(labelCol=..., maxIter=..., solver=...)
grille = ParamGridBuilder().addGrid(..., [...]).build()
validation = CrossValidator(
  estimator=regression,
  estimatorParamMaps=grille,
  evaluator=RegressionEvaluator(labelCol=...),
  numFolds=...,
)
modeles = []
mesurer(lambda: modeles.append(validation.fit(X)), essais=1)
print(modeles[-1].avgMetrics)
X.cache()
X.count()
mesurer(...)
X.unpersist()
```

### Solution

In [ ]:
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

X = (
  VectorAssembler(
    inputCols=["surface_reelle_bati_total", "nombre_pieces_principales"],
    outputCol="features",
  )
  .transform(ventes.filter("annee_mutation = 2022"))
  .select("features", "prix_m2")
)
regression = LinearRegression(labelCol="prix_m2", maxIter=20, solver="l-bfgs")
grille = ParamGridBuilder().addGrid(regression.regParam, [0.1, 1.0]).build()
validation = CrossValidator(
  estimator=regression,
  estimatorParamMaps=grille,
  evaluator=RegressionEvaluator(labelCol="prix_m2"),
  numFolds=3,
)
modeles = []
mesurer(lambda: modeles.append(validation.fit(X)), essais=1)
print(modeles[-1].avgMetrics)
X.cache()
X.count()
mesurer(lambda: modeles.append(validation.fit(X)), essais=1)
X.unpersist()
# Sept modèles : 3 plis × 2 réglages, puis le meilleur réentraîné sur tout X. Les
# deux réglages se valent (racine de l'erreur quadratique moyenne d'environ
# 3 500 €/m²). Sans cache, deux à trois fois plus long qu'avec (30 à 50 s contre
# 15 à 20 s), et le cache se remplit en une seconde.
# CrossValidator met lui-même en cache l'apprentissage et la validation de chaque
# pli, mais il les remplit à partir de X : sans cache, chacun des six remplissages
# et l'entraînement final relisent Parquet et refont la jointure, le filtre et
# l'assemblage des vecteurs.

## 8. Du code Python dans Spark

Gardez dans `surfaces` les transactions de surface bâtie positive, avec leur prix au mètre carré dans une colonne `prix_m2`. Comptez-les par tranche de prix : moins de 2 000 €/m², de 2 000 à 5 000, 5 000 et plus.

Écrivez la tranche d'abord comme une UDF Python, qui reçoit un prix et renvoie le nom de sa tranche, puis avec les fonctions de Spark (`F.when`). Vérifiez que les deux versions donnent les mêmes comptes, mesurez-les et comparez leurs plans : quel opérateur exécute l'UDF, et que devient la version native ?

Facultatif : calculez la médiane de `prix_m2` par département de deux façons, avec `F.median`, puis avec `applyInPandas`, qui applique une fonction pandas au `DataFrame` de chaque département. Comparez les durées et les plans : où passent les lignes d'un département avant d'arriver à pandas ?

**Documentation utile**

- [pyspark.sql.functions.when](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.when.html)
- [pyspark.sql.functions.median](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.median.html)
- [pyspark.sql.GroupedData.applyInPandas](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.GroupedData.applyInPandas.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Une UDF se déclare avec le décorateur `@F.udf("string")` sur une fonction Python ordinaire. `F.when(condition, valeur)` s'enchaîne : `F.when(...).when(...).otherwise(...)`, la première condition vraie l'emporte. La fonction d'`applyInPandas` reçoit un `pd.DataFrame` (les lignes d'un département) et en renvoie un, dont le schéma s'écrit en chaîne : `"code_departement string, mediane double"`.

```python
surfaces = dvf.filter(...).withColumn("prix_m2", ...)


@F.udf("string")
def tranche_udf(prix):
  if prix < 2000:
    return ...
  ...


tranche_native = F.when(..., ...).when(..., ...).otherwise(...)
versions = {"udf": tranche_udf("prix_m2"), "natif": tranche_native}
for nom, tranche in versions.items():
  print(nom, sorted(surfaces.groupBy(tranche.alias("tranche")).count().collect()))
  mesurer(lambda: surfaces.groupBy(tranche.alias("tranche")).count().collect())
  surfaces.groupBy(tranche.alias("tranche")).count().explain()


def mediane(departement: pd.DataFrame) -> pd.DataFrame:
  return pd.DataFrame(
    {
      "code_departement": [departement["code_departement"].iloc[0]],
      "mediane": [...],
    }
  )


par_departement = surfaces.select("code_departement", "prix_m2").groupBy(
  "code_departement"
)
medianes_natives = surfaces.groupBy(...).agg(F.median(...))
medianes_pandas = par_departement.applyInPandas(mediane, ...)
...
```

### Solution

In [ ]:
surfaces = dvf.filter("surface_reelle_bati_total > 0").withColumn(
  "prix_m2", F.col("valeur_fonciere") / F.col("surface_reelle_bati_total")
)


@F.udf("string")
def tranche_udf(prix):
  if prix < 2000:
    return "moins de 2 000"
  if prix < 5000:
    return "2 000 à 5 000"
  return "5 000 et plus"


tranche_native = (
  F.when(F.col("prix_m2") < 2000, "moins de 2 000")
  .when(F.col("prix_m2") < 5000, "2 000 à 5 000")
  .otherwise("5 000 et plus")
)
versions = {"udf": tranche_udf("prix_m2"), "natif": tranche_native}
for nom, tranche in versions.items():
  print(nom, sorted(surfaces.groupBy(tranche.alias("tranche")).count().collect()))
  mesurer(lambda: surfaces.groupBy(tranche.alias("tranche")).count().collect())
  surfaces.groupBy(tranche.alias("tranche")).count().explain()


def mediane(departement: pd.DataFrame) -> pd.DataFrame:
  return pd.DataFrame(
    {
      "code_departement": [departement["code_departement"].iloc[0]],
      "mediane": [departement["prix_m2"].median()],
    }
  )


par_departement = surfaces.select("code_departement", "prix_m2").groupBy(
  "code_departement"
)
medianes_natives = surfaces.groupBy("code_departement").agg(F.median("prix_m2"))
medianes_pandas = par_departement.applyInPandas(
  mediane, "code_departement string, mediane double"
)
for medianes in [medianes_natives, medianes_pandas]:
  mesurer(lambda: medianes.collect(), essais=1)
  medianes.explain()
print(dict(medianes_natives.collect())["75"], dict(medianes_pandas.collect())["75"])
# Mêmes comptes : 3,5 millions de transactions sous 2 000 €/m², 3,1 millions de 2 000 à
# 5 000, 0,8 million au-delà. L'UDF est deux à quatre fois plus lente : son plan
# l'exécute dans ArrowEvalPython (BatchEvalPython sur Spark 4.0, celui de Colab, où
# Arrow n'est pas le défaut des UDF), qui envoie les prix à un processus Python et en
# reçoit les tranches. La version native devient un CASE WHEN dans un Project, compilé
# avec le reste de l'étape, sans quitter la JVM. Facultatif : mêmes médianes (9 249 €/m²
# à Paris), en des durées proches, une dizaine de secondes : la médiane exacte rassemble
# et trie les prix de chaque département dans les deux cas. F.median en prépare une
# partie avant le shuffle (partial_percentile) ; dans le plan d'applyInPandas, un
# Exchange hashpartitioning(code_departement) envoie toutes les lignes d'un département
# à une même tâche, triées, puis FlatMapGroupsInPandas les passe à pandas d'un bloc : un
# groupe doit tenir dans la mémoire d'un processus Python. applyInPandas sert là où
# Spark n'a pas de fonction (un modèle par département), pas pour remplacer F.median.

## 9. Ramener des données au pilote

Gardez dans `annee_2022` les transactions de 2022, avec `valeur_fonciere`, `code_departement` et `surface_reelle_bati_total`. Combien de lignes compte-t-elle ? Mesurez sa conversion en `DataFrame` pandas (`toPandas`) avec Arrow désactivé (`spark.sql.execution.arrow.pyspark.enabled` à `"false"`), puis activé. Laissez Arrow activé.

**Documentation utile**

- [pyspark.sql.DataFrame.toPandas](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.toPandas.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

L'option se change en cours de session, comme celles du shuffle. Arrow est activé par défaut depuis Spark 4.2, mais pas sur Spark 4.0, celui de Colab : `print(spark.version)`, plus haut, dit lequel vous avez.

```python
annee_2022 = dvf.filter(...).select(...)
print(annee_2022.count())
for arrow in ["false", "true"]:
  spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", arrow)
  print(arrow, end=" ")
  mesurer(...)
```

### Solution

In [ ]:
annee_2022 = dvf.filter("annee_mutation = 2022").select(
  "valeur_fonciere", "code_departement", "surface_reelle_bati_total"
)
print(annee_2022.count())
for arrow in ["false", "true"]:
  spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", arrow)
  print(arrow, end=" ")
  mesurer(annee_2022.toPandas)
# 508 837 lignes. Sans Arrow : 2 s environ ; avec : 0,3 s, six fois moins.

## 10. Écrire pour la lecture suivante

Écrivez les transactions de 2022 (toutes les colonnes) en Parquet, partitionnées par département (`partitionBy`), de deux façons : dans `dvf-2022-petits` après un `repartition(20)`, comme après un shuffle qui laisse chaque département réparti entre ses partitions, puis dans `dvf-2022-departement` après un `repartition` sur `code_departement`. Mesurez chaque écriture (un essai), et comptez les fichiers Parquet et les dossiers de chacune.

Relisez chacune et mesurez sur elle la moyenne de `valeur_fonciere` par département (un essai) : d'où vient l'écart ?

Écrivez enfin la version par département dans `dvf-2022-plafond`, en plafonnant chaque fichier à 10 000 lignes (option `maxRecordsPerFile`). Combien de fichiers cette fois, et combien pour Paris ?

**Documentation utile**

- [pyspark.sql.DataFrameWriter.option](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.option.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`mode("overwrite")` remplace un dossier existant, ce qui permet de relancer la cellule. `pathlib.Path(dossier).glob("*/*.parquet")` liste les fichiers des sous-dossiers, et `glob("*/")` les sous-dossiers. Une option d'écriture se donne avant `parquet` : `.option("maxRecordsPerFile", 10000)`, ou plusieurs d'un coup avec `.options(**options)`.

```python
annee_2022_complete = dvf.filter(...)


def ecrire(transactions, dossier, **options):
  transactions.write.mode("overwrite").options(**options).partitionBy(...).parquet(
    dossier
  )


def compter(dossier):
  print(
    dossier,
    len(list(pathlib.Path(dossier).glob(...))),
    len(list(pathlib.Path(dossier).glob(...))),
  )


mesurer(
  lambda: ecrire(annee_2022_complete.repartition(...), "dvf-2022-petits"), essais=1
)
mesurer(lambda: ecrire(...), essais=1)
for dossier in ["dvf-2022-petits", "dvf-2022-departement"]:
  compter(dossier)
  mesurer(lambda: spark.read.parquet(dossier).groupBy(...).agg(...).collect(), essais=1)
ecrire(..., "dvf-2022-plafond", maxRecordsPerFile=...)
compter("dvf-2022-plafond")
print(len(list(pathlib.Path("dvf-2022-plafond/code_departement=75").glob(...))))
```

### Solution

In [ ]:
annee_2022_complete = dvf.filter("annee_mutation = 2022")


def ecrire(transactions, dossier, **options):
  transactions.write.mode("overwrite").options(**options).partitionBy(
    "code_departement"
  ).parquet(dossier)


def compter(dossier):
  print(
    dossier,
    len(list(pathlib.Path(dossier).glob("*/*.parquet"))),
    len(list(pathlib.Path(dossier).glob("*/"))),
  )


mesurer(
  lambda: ecrire(annee_2022_complete.repartition(20), "dvf-2022-petits"), essais=1
)
mesurer(
  lambda: ecrire(
    annee_2022_complete.repartition("code_departement"), "dvf-2022-departement"
  ),
  essais=1,
)
for dossier in ["dvf-2022-petits", "dvf-2022-departement"]:
  compter(dossier)
  mesurer(
    lambda: (
      spark.read.parquet(dossier)
      .groupBy("code_departement")
      .agg(F.avg("valeur_fonciere"))
      .collect()
    ),
    essais=1,
  )
ecrire(
  annee_2022_complete.repartition("code_departement"),
  "dvf-2022-plafond",
  maxRecordsPerFile=10000,
)
compter("dvf-2022-plafond")
print(len(list(pathlib.Path("dvf-2022-plafond/code_departement=75").glob("*.parquet"))))
# Après repartition(20), chacune des 20 tâches écrit un fichier par département
# qu'elle rencontre : 1 929 fichiers dans 97 dossiers, une vingtaine par dossier,
# de quelques Ko. Après repartition sur code_departement, chaque département est
# dans une seule tâche : 97 fichiers, un par dossier. L'écriture des petits fichiers
# prend plusieurs fois plus longtemps, et leur relecture aussi (une dizaine de
# secondes contre 3 s environ) : chaque fichier coûte une ouverture et la lecture de
# ses métadonnées, quelle que soit sa taille. Plafonné à 10 000 lignes, un
# département plus gros se coupe en plusieurs fichiers : 116 fichiers, dont 3 pour
# Paris (21 413 transactions). maxRecordsPerFile borne la taille des fichiers d'un
# gros volume, sans créer de petits fichiers pour les départements qui tiennent
# dans un seul.

## 11. Diagnostiquer une requête lente

La cellule suivante calcule le prix moyen des ventes de 2022 en Île-de-France, par département, d'une façon qui cumule plusieurs défauts. Mesurez `prix_moyen_lent` (un essai, il dure une minute environ), puis affichez ses étapes, les quantiles des tâches de la plus récente avec `taches`, et le plan de `ventes_idf_2022()` avec `explain("formatted")`.

Relevez chaque symptôme : que disent `ReadSchema`, `PartitionFilters` et `PushedFilters`, quel opérateur précède le filtre, quelles sont les clés de la jointure, et que ramène `prix_moyen_lent` au pilote ? Les étapes montrent-elles un déséquilibre ou un shuffle ?

Réécrivez la requête sans ces défauts, en partant de `avec_region` (exercice 5), vérifiez qu'elle donne les mêmes moyennes, et mesurez-la.

Les remèdes qui touchent à la mémoire se règlent, eux, au lancement : avec `spark.conf.isModifiable`, vérifiez lesquelles de `spark.executor.memory` et `spark.sql.shuffle.partitions` se changent en cours de session, et lisez la mémoire du pilote (`spark.conf.get`).

**Documentation utile**

- [pyspark.sql.conf.RuntimeConfig](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.conf.RuntimeConfig.html)

In [ ]:
table_regions = spark.read.csv("departements-region.csv", header=True)


@F.udf("boolean")
def vente_de_2022(nature, annee):
  return nature == "Vente" and annee == 2022


def ventes_idf_2022():
  return (
    dvf.join(table_regions, F.lpad("code_departement", 2, "0") == table_regions.num_dep)
    .filter(vente_de_2022("nature_mutation", "annee_mutation"))
    .filter("region_name = 'Île-de-France'")
  )


def prix_moyen_lent():
  ventes_idf = ventes_idf_2022().toPandas()
  return ventes_idf.groupby("dep_name")["valeur_fonciere"].mean()

In [ ]:
# Votre code ici

### Conseils supplémentaires

La frame « Symptômes et remèdes » liste ce qu'il faut chercher dans le plan quand *Stages* ne montre rien. Gardez le résultat de la requête lente dans une liste, comme le modèle de l'exercice 7, pour ne pas la relancer. Réécrite, la requête filtre par des comparaisons simples sur les colonnes, agrège dans Spark et ne ramène que le résultat ; `pd.Series(dict(lignes))` le met sous la forme de celui de `prix_moyen_lent`, et la différence des deux séries se calcule terme à terme.

```python
lents = []
mesurer(lambda: lents.append(prix_moyen_lent()), essais=1)
print(etapes())
print(taches(...))
ventes_idf_2022().explain("formatted")


def prix_moyen_rapide():
  lignes = avec_region.filter(...).groupBy(...).agg(F.avg(...)).collect()
  return pd.Series(dict(lignes))


print((lents[0] - prix_moyen_rapide()).abs().max())
mesurer(prix_moyen_rapide)
print(spark.conf.isModifiable(...), spark.conf.isModifiable(...))
print(spark.conf.get(...))
```

### Solution

In [ ]:
lents = []
mesurer(lambda: lents.append(prix_moyen_lent()), essais=1)
print(etapes())
print(taches(int(etapes(1)["stageId"].iloc[0])))
ventes_idf_2022().explain("formatted")


def prix_moyen_rapide():
  lignes = (
    avec_region.filter(
      "annee_mutation = 2022 AND nature_mutation = 'Vente' "
      "AND region_name = 'Île-de-France'"
    )
    .groupBy("dep_name")
    .agg(F.avg("valeur_fonciere"))
    .collect()
  )
  return pd.Series(dict(lignes))


print((lents[0] - prix_moyen_rapide()).abs().max())
mesurer(prix_moyen_rapide)
print(
  spark.conf.isModifiable("spark.executor.memory"),
  spark.conf.isModifiable("spark.sql.shuffle.partitions"),
)
print(spark.conf.get("spark.driver.memory"))
# Les symptômes, dans le plan. L'UDF vente_de_2022 tourne dans ArrowEvalPython
# (BatchEvalPython sur Spark 4.0) : les 10,8 millions de lignes passent par Python,
# et l'optimiseur ne voit pas ce qu'elle filtre. Donc PartitionFilters est vide,
# absent du plan (les neuf années sont lues, pour une seule utile), et PushedFilters
# ne garde que IsNotNull(code_departement) : le filtre sur la nature n'est pas
# poussé. ReadSchema liste toutes les colonnes (annee_mutation vient des dossiers),
# car toPandas les ramène toutes au pilote, 89 073 lignes de 38 colonnes, pour n'en
# moyenner qu'une en pandas. La clé de jointure est
# calculée par lpad, qui tronque « 971 » en « 97 » : sans effet sur
# l'Île-de-France, mais les départements d'outre-mer se perdraient sans erreur. La
# jointure est bien une diffusion. Les étapes ne montrent ni shuffle ni
# déséquilibre : une étape de 8 tâches, toutes lentes, une dizaine de secondes ou
# plus chacune. Réécrite (filtres SQL poussés au lecteur, année en filtre de partition,
# deux colonnes lues, moyenne calculée par Spark), la requête donne les mêmes
# moyennes (différence nulle, aux arrondis près) en 1 s environ, contre une minute.
# spark.executor.memory n'est pas modifiable en cours de session (False), au
# contraire des options SQL (True) : la mémoire, les cœurs et les exécuteurs se
# fixent au lancement, ici spark.driver.memory à 4g, celle du seul processus du
# mode local.

## Fin

Arrêtez la session, qui libère la mémoire de Spark.

In [ ]:
spark.stop()